# Airline Delay Classification with Weights & Biases

Refer to this [github project](https://github.com/JaHerbas/Predicting_Flight_Delays).

In [1]:
# --- Import modules ---
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import xgboost as xgb

import wandb
import wandb.sklearn

# Silence wandb output
os.environ["WANDB_SILENT"] = "true"

In [2]:
# --- Set logging level ---

import warnings
warnings.filterwarnings("ignore")

In [3]:
# --- Load dataset ---
DATA_DIR = os.environ.get("CIML26_DATA_DIR") 
filename = DATA_DIR + "/Airline_2016_only.csv"
df = pd.read_csv(filename)

In [4]:
# --- Create target ---

df['DELAYED'] = (df['ARR_DELAY'] > 0).astype(int)

In [5]:
# --- Choose features ---

columns_to_remove = [
    'DEP_TIME', 'DEP_DELAY', 'TAXI_OUT', 'WHEELS_OFF', 'WHEELS_ON', 'TAXI_IN',
    'ARR_TIME', 'ARR_DELAY', 'CANCELLED', 'CANCELLATION_CODE', 'DIVERTED',
    'ACTUAL_ELAPSED_TIME', 'AIR_TIME', 'CARRIER_DELAY', 'WEATHER_DELAY',
    'NAS_DELAY', 'SECURITY_DELAY', 'LATE_AIRCRAFT_DELAY', 'Unnamed: 27'
]
df.drop(columns=columns_to_remove, inplace=True)

In [6]:
df.columns

Index(['_c0', 'FL_DATE', 'OP_CARRIER', 'OP_CARRIER_FL_NUM', 'ORIGIN', 'DEST',
       'CRS_DEP_TIME', 'CRS_ARR_TIME', 'CRS_ELAPSED_TIME', 'DISTANCE',
       'DELAYED'],
      dtype='object')

In [7]:
# --- Preprocessing ---

categorical_features = ['OP_CARRIER', 'ORIGIN', 'DEST']
numeric_features = df.select_dtypes(include=['int64', 'float64']).columns.difference(['DELAYED'])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', SimpleImputer(strategy='median'), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

In [8]:
# --- Split dataset ---

X = df.drop(columns=['DELAYED', 'FL_DATE'])
y = df['DELAYED']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [9]:
print(X_train.shape)
print(X_test.shape)

(4494126, 9)
(1123532, 9)


In [10]:
# --- Define models ---

models = {
    'Logistic Regression': LogisticRegression(max_iter=100, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=4, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=20, max_depth=4, random_state=42),
    'XGBoost': xgb.XGBClassifier(eval_metric='logloss', n_estimators=20, max_depth=3, use_label_encoder=False, random_state=42)
}

In [11]:
# --- Initialize WandB ---

wandb.login()

True

## Since we silenced the Wandb output, you would have to navigate to wandb and go to the project

In [12]:
# --- Train and evaluate models ---

results = {}

for model_name, model in models.items():
    wandb.init(project="airline-delay-prediction_2", name=model_name, reinit=True)
    
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])
    pipeline.fit(X_train, y_train)

    # Train accuracy
    y_train_pred = pipeline.predict(X_train)
    train_accuracy = accuracy_score(y_train, y_train_pred)
    
    # Test accuracy
    y_test_pred = pipeline.predict(X_test)
    test_accuracy = accuracy_score(y_test, y_test_pred)
    
    report = classification_report(y_test, y_test_pred, output_dict=True)

    # Log model, parameters, and metrics
    wandb.log({
        "train_accuracy": train_accuracy,
        "test_accuracy": test_accuracy
    })
    metrics_to_log = {}
    for label, metrics in report.items():
        if isinstance(metrics, dict):
            for metric_name, metric_value in metrics.items():
                metrics_to_log[f"{label}_{metric_name}"] = metric_value

    wandb.log(metrics_to_log)

    results[model_name] = {'accuracy': test_accuracy}
    
    wandb.finish()

In [13]:
metrics_to_log

{'0_precision': 0.6758360151632616,
 '0_recall': 0.9745398978593354,
 '0_f1-score': 0.7981565363258513,
 '0_support': 746030.0,
 '1_precision': 0.6024197262109097,
 '1_recall': 0.07623800668605729,
 '1_f1-score': 0.13534739792511216,
 '1_support': 377502.0,
 'macro avg_precision': 0.6391278706870857,
 'macro avg_recall': 0.5253889522726963,
 'macro avg_f1-score': 0.4667519671254817,
 'macro avg_support': 1123532.0,
 'weighted avg_precision': 0.6511684525908642,
 'weighted avg_recall': 0.6727142618100775,
 'weighted avg_f1-score': 0.5754554691870819,
 'weighted avg_support': 1123532.0}

In [14]:
# --- Display results ---

results_df = pd.DataFrame(results).T
results_df

,accuracy
Logistic Regression,0.664003
Decision Tree,0.664067
Random Forest,0.664004
XGBoost,0.672714
